In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import json
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from datasets import Dataset
from huggingface_hub import login, hf_hub_download
from transformers import PretrainedConfig, PreTrainedTokenizerFast
from transformers.modeling_outputs import SequenceClassifierOutput
from safetensors.torch import load_file
from sklearn.metrics import precision_recall_fscore_support
import zipfile
from tqdm.auto import tqdm

In [ ]:
# ==========================================
# 1. AUTHENTICATION & CONFIGURATION
# ==========================================
login(token=os.environ["HF_TOKEN"])

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
HYBRID_ENCODER_REPO = "A-240242/legal-hybridencoder-checkpoints"
HF_TOKENIZER_REPO = "A-240242/legal-modern-decoder-checkpoints"
OUTPUT_DIR = "./phase4_hybrid_encoder_lorma"

def load_cjpe_file(filename):
    path = hf_hub_download(
        repo_id=HF_DATASET_REPO, 
        filename=f"Phase_4_CJPE/{filename}", 
        repo_type="dataset", 
        token=os.environ["HF_TOKEN"]
    )
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

print("Downloading Phase 4 CJPE Data...")
train_data = load_cjpe_file("Train/train.json")
dev_data = load_cjpe_file("Train/dev.json")

# Toggle debug mode if a rapid sanity check is needed
DEBUG_MODE = False
if DEBUG_MODE:
    print("⚠️ DEBUG MODE ACTIVE: Subsampling dataset.")
    train_data = train_data[:100]
    dev_data = dev_data[:20]



In [ ]:
# ==========================================
# 2. TOKENIZER & DUAL-ANCHOR CHUNKING
# ==========================================
tokenizer = PreTrainedTokenizerFast.from_pretrained(HF_TOKENIZER_REPO, token=os.environ["HF_TOKEN"])
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def prepare_dataset_records(raw_records, tokenizer, max_length=1024):
    input_ids_list = []
    attention_mask_list = []
    labels_list = []
    
    for record in raw_records:
        doc_text = record.get("text", "")
        label = record.get("label", 0)
        
        tokenized = tokenizer(doc_text, truncation=False, return_attention_mask=False)
        full_ids = tokenized["input_ids"]
        
        if len(full_ids) <= max_length:
            chunks = [full_ids]
        else:
            first_chunk = full_ids[:max_length]
            last_chunk = full_ids[-max_length:]
            chunks = [first_chunk, last_chunk]
            
        for chunk in chunks:
            input_ids_list.append(chunk)
            attention_mask_list.append([1] * len(chunk))
            labels_list.append(label)
            
    return {
        "input_ids": input_ids_list,
        "attention_mask": attention_mask_list,
        "labels": labels_list
    }

print("Processing Dataset Features...")
train_features = prepare_dataset_records(train_data, tokenizer)
dev_features = prepare_dataset_records(dev_data, tokenizer)

train_dataset = Dataset.from_dict(train_features)
dev_dataset = Dataset.from_dict(dev_features)

def custom_collate_fn(batch):
    input_ids = [torch.tensor(item["input_ids"], dtype=torch.long) for item in batch]
    attention_mask = [torch.tensor(item["attention_mask"], dtype=torch.long) for item in batch]
    labels = [torch.tensor(item["labels"], dtype=torch.long) for item in batch]
    
    input_ids = torch.nn.utils.rnn.pad_sequence(input_ids, batch_first=True, padding_value=tokenizer.pad_token_id)
    attention_mask = torch.nn.utils.rnn.pad_sequence(attention_mask, batch_first=True, padding_value=0)
    labels = torch.stack(labels)
    
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=custom_collate_fn)
dev_loader = DataLoader(dev_dataset, batch_size=4, shuffle=False, collate_fn=custom_collate_fn)



In [ ]:
# ==========================================
# 3. HYBRID ENCODER LORMA ARCHITECTURE
# ==========================================
class LORMALinear(nn.Module):
    def __init__(self, in_features, out_features, rank=16, alpha=32, dropout_p=0.05):
        super().__init__()
        self.base_layer = nn.Linear(in_features, out_features, bias=False)
        self.base_layer.weight.requires_grad = False
        
        self.lorma_A = nn.Linear(in_features, rank, bias=False)
        self.lorma_B = nn.Linear(rank, out_features, bias=False)
        
        self.scaling = alpha / rank
        self.dropout = nn.Dropout(p=dropout_p)
        
        nn.init.kaiming_uniform_(self.lorma_A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lorma_B.weight)

    def forward(self, x):
        base_output = self.base_layer(x)
        lorma_output = self.lorma_B(self.lorma_A(self.dropout(x))) * self.scaling
        return base_output + lorma_output

class HybridLORMALocalBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.bilstm = nn.LSTM(
            input_size=config.hidden_size,
            hidden_size=config.hidden_size // 2,
            batch_first=True,
            bidirectional=True
        )

    def forward(self, x):
        local_features, _ = self.bilstm(x)
        return local_features

class HybridLORMAAttention(nn.Module):
    def __init__(self, config, rank=16, alpha=32):
        super().__init__()
        self.hidden_size = config.hidden_size
        self.num_heads = config.num_attention_heads
        self.head_dim = self.hidden_size // self.num_heads
        
        self.q_proj = LORMALinear(self.hidden_size, self.num_heads * self.head_dim, rank=rank, alpha=alpha)
        self.k_proj = LORMALinear(self.hidden_size, self.num_heads * self.head_dim, rank=rank, alpha=alpha)
        self.v_proj = LORMALinear(self.hidden_size, self.num_heads * self.head_dim, rank=rank, alpha=alpha)
        self.o_proj = LORMALinear(self.num_heads * self.head_dim, self.hidden_size, rank=rank, alpha=alpha)

    def forward(self, hidden_states, attention_mask=None):
        bsz, q_len, _ = hidden_states.size()
        
        q = self.q_proj(hidden_states).view(bsz, q_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(hidden_states).view(bsz, q_len, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(hidden_states).view(bsz, q_len, self.num_heads, self.head_dim).transpose(1, 2)
        
        attn_mask = None
        if attention_mask is not None:
            attn_mask = attention_mask.unsqueeze(1).unsqueeze(2)
            attn_mask = attn_mask.expand(bsz, self.num_heads, q_len, q_len).bool()

        attn_output = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask, is_causal=False)
        attn_output = attn_output.transpose(1, 2).contiguous().view(bsz, q_len, self.hidden_size)
        
        return self.o_proj(attn_output)

class HybridEncoderLORMALayer(nn.Module):
    def __init__(self, config, rank=16, alpha=32):
        super().__init__()
        self.local_block = HybridLORMALocalBlock(config)
        self.global_attn = HybridLORMAAttention(config, rank=rank, alpha=alpha)
        
        self.ffn_up = LORMALinear(config.hidden_size, config.intermediate_size, rank=rank, alpha=alpha)
        self.ffn_down = LORMALinear(config.intermediate_size, config.hidden_size, rank=rank, alpha=alpha)
        
        self.norm1 = nn.LayerNorm(config.hidden_size)
        self.norm2 = nn.LayerNorm(config.hidden_size)

    def forward(self, hidden_states, attention_mask=None):
        local_out = self.local_block(hidden_states)
        hidden_states = hidden_states + local_out
        
        attn_out = self.global_attn(self.norm1(hidden_states), attention_mask)
        hidden_states = hidden_states + attn_out
        
        ffn_out = self.ffn_down(F.gelu(self.ffn_up(self.norm2(hidden_states))))
        hidden_states = hidden_states + ffn_out
        
        return hidden_states

class HybridEncoderLORMAForSequenceClassification(nn.Module):
    def __init__(self, config, num_labels, rank=16, alpha=32):
        super().__init__()
        self.config = config
        self.num_labels = num_labels
        
        self.embed_tokens = nn.Embedding(config.vocab_size, config.hidden_size, padding_idx=config.pad_token_id)
        self.layers = nn.ModuleList([HybridEncoderLORMALayer(config, rank=rank, alpha=alpha) for _ in range(config.num_hidden_layers)])
        self.norm = nn.LayerNorm(config.hidden_size)
        self.score = nn.Linear(config.hidden_size, num_labels, bias=False)

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        hidden_states = self.embed_tokens(input_ids)
        
        for layer in self.layers:
            hidden_states = layer(hidden_states, attention_mask)
            
        hidden_states = self.norm(hidden_states)
        
        if attention_mask is not None:
            mask_expanded = attention_mask.unsqueeze(-1).expand(hidden_states.size()).float()
            sum_embeddings = torch.sum(hidden_states * mask_expanded, 1)
            sum_mask = torch.clamp(mask_expanded.sum(1), min=1e-9)
            pooled_logits = sum_embeddings / sum_mask
        else:
            pooled_logits = hidden_states.mean(dim=1)

        logits = self.score(pooled_logits)
        
        loss = None
        if labels is not None:
            loss_fn = nn.CrossEntropyLoss()
            loss = loss_fn(logits.view(-1, self.num_labels), labels.view(-1))
            
        return SequenceClassifierOutput(loss=loss, logits=logits)



In [ ]:
# ==========================================
# 4. INITIALIZATION, MAPPING & FREEZING
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Fetching Phase 1 Pretrained Hybrid Encoder Checkpoint...")
hybrid_config = PretrainedConfig(
    vocab_size=tokenizer.vocab_size,
    pad_token_id=tokenizer.pad_token_id,
    hidden_size=768,
    num_hidden_layers=6,
    num_attention_heads=12,
    intermediate_size=2048
)

print("Initializing Hybrid Encoder with LORMA Adapters...")
model = HybridEncoderLORMAForSequenceClassification(hybrid_config, num_labels=2, rank=16, alpha=32).to(device)

# Load base weights from Phase 1 repository
model_weight_path = hf_hub_download(repo_id=HYBRID_ENCODER_REPO, filename="model.safetensors", token=HF_TOKEN)
state_dict = load_file(model_weight_path)

mapped_state_dict = {}
for key, tensor in state_dict.items():
    if "proj.weight" in key or "ffn_up.weight" in key or "ffn_down.weight" in key:
        mapped_state_dict[key.replace(".weight", ".base_layer.weight")] = tensor
    else:
        mapped_state_dict[key] = tensor

model.load_state_dict(mapped_state_dict, strict=False)

# Freeze backbone, unfreeze LORMA parameters and classification score head
for param in model.parameters(): 
    param.requires_grad = False

for name, param in model.named_parameters():
    if "lorma_A" in name or "lorma_B" in name or "score" in name:
        param.requires_grad = True

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable LORMA Parameters: {trainable_params:,}")

In [ ]:
# ==========================================
# 5. TRAINING LOOP
# ==========================================
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=3e-4)
criterion = nn.CrossEntropyLoss()

EPOCHS = 3
print("\nStarting Hybrid Encoder LORMA Training Loop for Phase 4...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    
    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        
    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {total_loss / len(train_loader):.4f}")
    
    # Validation Evaluation
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.inference_mode():
        for batch in dev_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = outputs.logits.argmax(dim=-1).cpu().tolist()
            
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().tolist())
            
    precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average="macro", zero_division=0)
    print(f"Validation Macro-F1: {f1:.4f} (Precision: {precision:.4f}, Recall: {recall:.4f})")



In [ ]:
# ==========================================
# 6. TEST INFERENCE & SUBMISSION GENERATION
# ==========================================
print("\nLoading Test Data for Submission Generation...")
test_file_path = hf_hub_download(
    repo_id=HF_DATASET_REPO, 
    filename="Phase_4_CJPE/Test/test.json", 
    repo_type="dataset", 
    token=HF_TOKEN
)

with open(test_file_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

model.eval()
predictions = []

print(f"Running inference on {len(test_data)} test examples...")
for case in tqdm(test_data, desc="Predicting CJPE Cases"):
    case_id = case.get("id")
    case_text = case.get("text", "")
    
    encoded = tokenizer(
        case_text, 
        truncation=False, 
        return_attention_mask=False
    )
    full_ids = encoded["input_ids"]
    chunk = full_ids[:1024]
    
    input_ids = torch.tensor([chunk], dtype=torch.long).to(device)
    attention_mask = torch.tensor([[1] * len(chunk)], dtype=torch.long).to(device)
    
    with torch.inference_mode():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        pred_label = int(outputs.logits.argmax(dim=-1).item())
        
    predictions.append({
        "id": case_id,
        "pred": pred_label
    })

output_dir = "./phase4_hybrid_encoder_lorma_cjpe"
os.makedirs(output_dir, exist_ok=True)
pred_path = os.path.join(output_dir, "prediction.json")
sub_zip_path = os.path.join(output_dir, "submission.zip")

with open(pred_path, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

with zipfile.ZipFile(sub_zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(pred_path, arcname="prediction.json")

print(f"\nPhase 4 CJPE Prediction complete! Generated {len(predictions)} predictions.")
print(f"Submission saved to: {sub_zip_path}")